# SitePulse — Feature Engineering Validation

## Objective

This notebook validates the engineered features created by the SitePulse feature pipeline.

The analysis focuses on:

- schedule performance features
- normalized workforce metrics
- fuel efficiency
- material delivery efficiency
- cost efficiency
- rolling 7-day operational indicators
- schedule status categories
- schedule risk indicators

The objective is to confirm that the engineered variables are logically consistent and suitable for downstream analytics and machine learning.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

In [2]:
features = pd.read_csv(
    PROCESSED_DATA_DIR / "operations_features.csv",
    parse_dates=["date"],
)

print("Shape:", features.shape)

Shape: (1390, 35)


In [3]:
engineered_features = [
    "schedule_variance_pct",
    "daily_progress_gain",
    "workforce_ratio",
    "workforce_variance_pct",
    "fuel_per_equipment_hour",
    "material_delivery_ratio",
    "progress_per_material_t",
    "cost_per_progress_point",
    "delay_ratio",
    "rolling_7d_fuel",
    "rolling_7d_progress",
    "rolling_7d_delay",
    "schedule_status",
    "schedule_risk_flag",
]

features[engineered_features].head()

,schedule_variance_pct,daily_progress_gain,workforce_ratio,workforce_variance_pct,fuel_per_equipment_hour,material_delivery_ratio,progress_per_material_t,cost_per_progress_point,delay_ratio,rolling_7d_fuel,rolling_7d_progress,rolling_7d_delay,schedule_status,schedule_risk_flag
0,-0.03,0.39,0.952941,-4.705882,15.375670,1.124113,0.046099,18618.102564,0.022500,487.8700,0.390000,0.270000,On Track,0
1,-0.16,0.28,1.011765,1.176471,13.918224,1.006380,0.044657,25119.750000,0.205000,392.8600,0.335000,1.365000,On Track,0
2,-0.30,0.28,1.070588,7.058824,15.291226,1.062963,0.051852,27370.714286,0.293333,372.8700,0.316667,2.083333,On Track,0
3,-0.56,0.16,0.929412,-7.058824,14.989828,0.529412,0.047059,42478.562500,0.386667,375.4375,0.277500,2.722500,On Track,1
4,-0.71,0.26,1.035294,3.529412,16.300062,1.311741,0.052632,23848.923077,0.368333,352.6080,0.274000,3.062000,On Track,1


In [4]:
schedule_check = features[
    [
        "planned_progress_pct",
        "actual_progress_pct",
        "schedule_variance_pct",
    ]
].head(10).copy()

schedule_check["manual_calculation"] = (
    schedule_check["actual_progress_pct"]
    - schedule_check["planned_progress_pct"]
)

schedule_check

,planned_progress_pct,actual_progress_pct,schedule_variance_pct,manual_calculation
0,0.42,0.39,-0.03,-0.03
1,0.83,0.67,-0.16,-0.16
2,1.25,0.95,-0.30,-0.30
3,1.67,1.11,-0.56,-0.56
4,2.08,1.37,-0.71,-0.71
5,2.50,1.69,-0.81,-0.81
6,2.92,1.90,-1.02,-1.02
7,3.33,2.13,-1.20,-1.20
8,3.75,2.52,-1.23,-1.23
9,4.17,2.85,-1.32,-1.32


In [5]:
workforce_check = features[
    [
        "workers",
        "base_workers",
        "workforce_ratio",
        "workforce_variance_pct",
    ]
].head(10).copy()

workforce_check["manual_ratio"] = (
    workforce_check["workers"]
    / workforce_check["base_workers"]
)

workforce_check

,workers,base_workers,workforce_ratio,workforce_variance_pct,manual_ratio
0,81,85,0.952941,-4.705882,0.952941
1,86,85,1.011765,1.176471,1.011765
2,91,85,1.070588,7.058824,1.070588
3,79,85,0.929412,-7.058824,0.929412
4,88,85,1.035294,3.529412,1.035294
5,85,85,1.000000,0.000000,1.000000
6,81,85,0.952941,-4.705882,0.952941
7,79,85,0.929412,-7.058824,0.929412
8,78,85,0.917647,-8.235294,0.917647
9,87,85,1.023529,2.352941,1.023529


In [6]:
features["workforce_ratio"].describe()

count    1390.000000
mean        0.990887
std         0.096892
min         0.682353
25%         0.927273
50%         0.988562
75%         1.054545
max         1.307692
Name: workforce_ratio, dtype: float64

In [7]:
fuel_check = features[
    [
        "fuel_consumption_l",
        "equipment_hours",
        "fuel_per_equipment_hour",
        "fuel_anomaly_flag",
    ]
].head(10).copy()

fuel_check["manual_fuel_rate"] = (
    fuel_check["fuel_consumption_l"]
    / fuel_check["equipment_hours"]
)

fuel_check

,fuel_consumption_l,equipment_hours,fuel_per_equipment_hour,fuel_anomaly_flag,manual_fuel_rate
0,487.87,31.73,15.375670,0,15.375670
1,297.85,21.40,13.918224,0,13.918224
2,332.89,21.77,15.291226,0,15.291226
3,383.14,25.56,14.989828,0,14.989828
4,261.29,16.03,16.300062,0,16.300062
5,422.95,27.75,15.241441,0,15.241441
6,209.62,14.56,14.396978,0,14.396978
7,425.82,29.62,14.376097,0,14.376097
8,474.06,30.92,15.331824,0,15.331824
9,427.50,26.78,15.963406,0,15.963406


In [8]:
fuel_efficiency_check = (
    features
    .groupby(
        "fuel_anomaly_flag",
        as_index=False,
    )
    .agg(
        average_fuel_per_hour=(
            "fuel_per_equipment_hour",
            "mean",
        ),
        median_fuel_per_hour=(
            "fuel_per_equipment_hour",
            "median",
        ),
    )
)

fuel_efficiency_check.round(2)

,fuel_anomaly_flag,average_fuel_per_hour,median_fuel_per_hour
0,0,15.31,15.20
1,1,24.78,24.49


In [9]:
material_check = features[
    [
        "material_delivered_t",
        "material_used_t",
        "material_delivery_ratio",
        "material_shortage_flag",
    ]
].head(10)

material_check

,material_delivered_t,material_used_t,material_delivery_ratio,material_shortage_flag
0,9.51,8.46,1.124113,0
1,6.31,6.27,1.006380,0
2,5.74,5.40,1.062963,0
3,1.80,3.40,0.529412,1
4,6.48,4.94,1.311741,0
5,8.82,7.69,1.146944,0
6,5.20,4.54,1.145374,0
7,3.76,5.86,0.641638,1
8,11.56,11.94,0.968174,0
9,6.48,6.19,1.046850,0


In [10]:
features.groupby(

    "material_shortage_flag"

)["material_delivery_ratio"].mean()

material_shortage_flag
0    1.151783
1    0.556907
Name: material_delivery_ratio, dtype: float64

In [11]:
first_project = features[
    features["project_id"] == "PRJ001"
][
    [
        "date",
        "fuel_consumption_l",
        "rolling_7d_fuel",
        "daily_progress_gain",
        "rolling_7d_progress",
        "delay_hours",
        "rolling_7d_delay",
    ]
].head(10)

first_project

,date,fuel_consumption_l,rolling_7d_fuel,daily_progress_gain,rolling_7d_progress,delay_hours,rolling_7d_delay
0,2026-01-05,487.87,487.870000,0.39,0.390000,0.27,0.270000
1,2026-01-06,297.85,392.860000,0.28,0.335000,2.46,1.365000
2,2026-01-07,332.89,372.870000,0.28,0.316667,3.52,2.083333
3,2026-01-08,383.14,375.437500,0.16,0.277500,4.64,2.722500
4,2026-01-09,261.29,352.608000,0.26,0.274000,4.42,3.062000
5,2026-01-10,422.95,364.331667,0.32,0.281667,1.92,2.871667
6,2026-01-11,209.62,342.230000,0.21,0.271429,4.21,3.062857
7,2026-01-12,425.82,333.365714,0.23,0.248571,4.80,3.710000
8,2026-01-13,474.06,358.538571,0.39,0.264286,0.00,3.358571
9,2026-01-14,427.50,372.054286,0.33,0.271429,2.33,3.188571


In [12]:
schedule_status_summary = (
    features["schedule_status"]
    .value_counts()
    .to_frame("count")
)

schedule_status_summary["percentage"] = (
    schedule_status_summary["count"]
    / len(features)
    * 100
).round(2)

schedule_status_summary

,count,percentage
schedule_status,,
On Track,793,57.05
Behind,456,32.81
Ahead,141,10.14


In [13]:
risk_summary = (
    features["schedule_risk_flag"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)

risk_summary["percentage"] = (
    risk_summary["count"]
    / len(features)
    * 100
).round(2)

risk_summary

,count,percentage
schedule_risk_flag,,
0,1162,83.6
1,228,16.4


In [14]:
features.groupby(
    "schedule_risk_flag"
)[
    [
        "schedule_variance_pct",
        "rolling_7d_delay",
        "daily_progress_gain",
    ]
].mean().round(2)

,schedule_variance_pct,rolling_7d_delay,daily_progress_gain
schedule_risk_flag,,,
0,-0.35,1.09,0.43
1,-8.08,1.55,0.36


In [15]:
numeric_engineered_features = [
    column
    for column in engineered_features
    if pd.api.types.is_numeric_dtype(
        features[column]
    )
]

feature_quality = pd.DataFrame(
    {
        "missing_values": (
            features[
                numeric_engineered_features
            ]
            .isna()
            .sum()
        ),

        "infinite_values": (
            features[
                numeric_engineered_features
            ]
            .apply(
                lambda column:
                    np.isinf(column).sum()
            )
        ),
    }
)

feature_quality

,missing_values,infinite_values
schedule_variance_pct,0,0
daily_progress_gain,0,0
workforce_ratio,0,0
workforce_variance_pct,0,0
fuel_per_equipment_hour,0,0
material_delivery_ratio,0,0
progress_per_material_t,0,0
cost_per_progress_point,16,0
delay_ratio,0,0
rolling_7d_fuel,0,0
